# Complete Machine Learning Pipeline

A practical, repeatable sequence for taking a dataset from its original source to a deployed model. The examples below use **scikit-learn** and a classification target; notes explain how to adapt the workflow for regression. Replace `data.csv` and `target_column` with your own file and target.

> **Important:** Split the data before fitting imputers, encoders, scalers, feature selectors, or models. Put preprocessing and the estimator in a pipeline so cross-validation and test evaluation do not learn from held-out data.

## 1. Data collection and loading

Collect data from a CSV/Excel file, a database, an API, or a carefully documented experiment. Check that the source is permitted, trustworthy, and relevant to the question. Keep an untouched copy of the raw data where possible.

```python
import pandas as pd

df = pd.read_csv("data.csv")
# Other common options:
# df = pd.read_excel("data.xlsx")
# df = pd.read_sql(query, connection)
```

## 2. Data understanding (EDA)

Exploratory data analysis (EDA) helps reveal data types, ranges, class balance, missing values, duplicates, and possible data-quality issues before modeling.

```python
df.head()                 # Preview rows
df.sample(5)              # Inspect a random sample
df.shape                  # Number of rows and columns
df.info()                 # Column types and non-null counts
df.describe()             # Numeric summaries
df.describe(include="all") #describe all the column with text or sting values too
df.isna().sum()           # Missing values by column
df.duplicated().sum()     # Duplicate rows
df["target_column"].value_counts(dropna=False)  # Target distribution
```

Visual methods are useful too: histograms/box plots for numeric distributions, bar charts for categories, scatter plots for relationships, and a correlation heatmap for numeric variables. Investigate unusual values rather than automatically deleting them; they may be valid observations.

## 3. Data cleaning

Correct invalid values and inconsistent labels using knowledge of the data. Remove duplicates only when they are genuinely repeated records. Choose a missing-value strategy based on the meaning of the column: remove a small number of unusable rows, impute numeric values with a median/mean, or fill categorical values with the most frequent value or an explicit `Unknown` category.

For example, inspect duplicates before removing them:

```python
df = df.drop_duplicates().copy()
```

The reusable modeling example below imputes missing feature values inside a pipeline. This is preferable to calculating imputation values on the entire dataset before splitting.

## 4. Data preprocessing

Machine-learning algorithms generally need numeric inputs. Common methods include:

- **Scaling:** `StandardScaler` centers and scales numeric columns; `MinMaxScaler` maps them to a fixed range. Scaling is important for distance- and gradient-based methods (such as KNN, SVM, and logistic regression), but usually unnecessary for tree-based models.
- **Encoding categories:** `OneHotEncoder` is suitable for nominal categories (no natural order). `OrdinalEncoder` is for categories with a real order. Avoid using `LabelEncoder` on ordinary input features because it can imply a false ranking; it is intended mainly for target labels.
- **Imputation:** `SimpleImputer` can use a median for numeric columns and the most frequent value for categorical columns.

A `ColumnTransformer` applies the right transformations to each column group. The complete setup appears in the end-to-end example below.

## 5. Feature selection and feature engineering

**Feature engineering** creates useful inputs from existing information, guided by the problem (for example, extracting month from a date or calculating price per unit). Apply the same transformation at training and prediction time; implement it in a transformer/pipeline for production use.

**Feature selection** removes irrelevant or redundant inputs. Methods include domain knowledge, correlation checks, `SelectKBest`, recursive feature elimination (`RFE`), and model-based importance. Perform selection using training data or inside cross-validation—not on the full dataset before the test split.

## 6. Train-test split

Keep a test set aside to estimate performance on unseen data. A common split is 80% training and 20% testing. For classification, `stratify=y` preserves class proportions when each class has enough examples. For time-dependent data, split chronologically instead of shuffling; for repeated people/devices, consider group-based splitting to prevent related records appearing in both sets.

```python
from sklearn.model_selection import train_test_split

X = df.drop(columns="target_column")
y = df["target_column"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

## 7. Model selection

Start with a simple baseline, then compare suitable models using the same cross-validation splits and metric. Possible classification models include `DummyClassifier` (baseline), `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`, and `SVC`. For regression, options include `DummyRegressor`, `LinearRegression`, `DecisionTreeRegressor`, and `RandomForestRegressor`.

Consider interpretability, dataset size, class imbalance, training time, and the cost of different errors—not accuracy alone. Cross-validation (for example, `cross_val_score` or `StratifiedKFold`) is more reliable than choosing a model from one train/test result.

## 8. Model training

Call `.fit(X_train, y_train)` to learn from the training set. In the example, the model pipeline first preprocesses columns and then fits logistic regression. The preprocessor learns imputation, scaling, and category encodings from training data only.

## 9. Model prediction

`predict(X)` returns predicted classes (or numeric values for regression). Classification models may also provide `predict_proba(X)` for class probabilities; these probabilities are not automatically calibrated and a decision threshold other than 0.5 may be appropriate for the application.

## 10. Model evaluation

Evaluate on the untouched test set after model choices are complete.

- **Classification:** confusion matrix, precision, recall, F1-score, and (when appropriate) ROC-AUC. Accuracy can be misleading for imbalanced classes.
- **Regression:** MAE, RMSE, and R-squared. MAE is in target units; RMSE penalizes larger errors more heavily.

Inspect errors and compare them with the baseline. Do not repeatedly tune decisions against the test set; doing so turns it into training data.

## 11. Hyperparameter tuning

Hyperparameters are settings chosen before/during training (for example, regularization strength or tree depth). `GridSearchCV` tests every specified combination; `RandomizedSearchCV` samples a fixed number and can be faster for large search spaces. Tune with cross-validation on the training data and select a metric relevant to the task. Refit the selected pipeline on all training data, then evaluate it once on the test set.

## 12. Deployment and monitoring

Save the **whole fitted pipeline**, not just the estimator, so prediction data receives exactly the same preprocessing. Load it in a controlled environment and validate incoming columns and data types. Version the model, preprocessing, training data reference, and dependencies. Monitor input drift and real-world performance, and retrain only with an appropriate validation process.

## End-to-end classification template

Run this after replacing the file path and target column. It assumes a categorical classification target.

```python
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data.csv").drop_duplicates().copy()
target = "target_column"
df = df.dropna(subset=[target])  # A row without a known label cannot train the model.
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
numeric_features = X_train.select_dtypes(include=np.number).columns
categorical_features = X_train.select_dtypes(exclude=np.number).columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions, zero_division=0))

# Tune on training data only. Compare other models and metrics as appropriate.
search = GridSearchCV(
    pipeline,
    param_grid={"model__C": [0.1, 1.0, 10.0]},
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1,
)
search.fit(X_train, y_train)
best_pipeline = search.best_estimator_
print("Best parameters:", search.best_params_)
print("Test score:", search.score(X_test, y_test))

# Save/load the complete pipeline. Only load trusted model files.
joblib.dump(best_pipeline, "model_pipeline.joblib")
loaded_pipeline = joblib.load("model_pipeline.joblib")
new_predictions = loaded_pipeline.predict(X_test.iloc[:3])
print(new_predictions)
```

For a **regression** task, use the same preprocessing structure but do not use `stratify=y`; select a regressor such as `RandomForestRegressor` or `LinearRegression`, tune a regression metric (for example `neg_mean_absolute_error`), and evaluate with `mean_absolute_error`, `root_mean_squared_error` (or the square root of MSE on older scikit-learn versions), and `r2_score`.
